# 레슨 05 — 데이터 정제 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 이번 강의는 코드 한 줄보다 정제 기준이 중요하다. 각 정답에는 코드와 함께 `왜 이 코드가 정답인지` 설명을 포함한다.

## 환경 셀

In [ ]:
import os
import pandas as pd
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/05/data"
else:
    DATA_BASE = "./data"
OUTPUT_PATH = "./clean_survey.csv"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 더러운 설문 데이터 불러오기

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/dirty_survey.csv")

print("shape:", df.shape)
print("columns:", list(df.columns))
print("dtypes:")
print(df.dtypes)
print("앞 5행:")
print(df.head())
print("뒤 3행:")
print(df.tail(3))
print("info:")
df.info()

rows, cols = df.shape
print(f"원본 데이터는 {rows:,}행 {cols}열입니다.")

### 왜 이 코드가 정답인지

정제의 첫 단계는 원본 구조를 바꾸지 않고 관찰하는 것이다. `shape`, `columns`, `dtypes`, `head`, `tail`, `info` 는 데이터의 크기, 열 이름, 자료형, 결측 가능성을 빠르게 보여준다. 원본 행 수와 열 수를 먼저 기록해야 중복 제거와 정제 후 결과가 얼마나 달라졌는지 비교할 수 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 원본 행 수 | 528 |
| 원본 열 수 | 9 |
| 주요 결측 열 | gender, satisfaction, screen_time, club |

---

## 문제 2 정답 — 결측치 현황 진단

In [ ]:
missing_by_col = df.isna().sum()
total_missing = missing_by_col.sum()
total_cells = df.shape[0] * df.shape[1]
missing_ratio = total_missing / total_cells * 100
missing_columns = missing_by_col[missing_by_col > 0]

print("열별 결측 개수:")
print(missing_by_col)
print("전체 결측 셀 수:", total_missing)
print("전체 결측 비율:", f"{missing_ratio:.2f}%")
print("결측이 있는 열:")
print(missing_columns)
print("결측이 있는 행 예시:")
print(df[df.isna().any(axis=1)].head())

### 왜 이 코드가 정답인지

`isna().sum()` 은 열별 결측 개수를 숫자로 보여준다. 전체 결측 셀 수와 비율을 계산하면 데이터 품질을 하나의 지표로 볼 수 있다. 결측이 있는 열만 따로 보면 정제 대상 열이 명확해진다. 결측 행 예시를 출력하면 숫자만으로 알 수 없는 실제 패턴을 확인할 수 있다.

**예상 핵심값**

```text
전체 결측 셀 수: 244
전체 결측 비율: 약 5.13%
```

---

## 문제 3 정답 — 중복 행 확인과 제거

In [ ]:
duplicate_all = df.duplicated().sum()
duplicate_student = df.duplicated("student_id").sum()

print("전체 행 기준 중복:", duplicate_all)
print("student_id 기준 중복:", duplicate_student)
print("중복 행 예시:")
print(df[df.duplicated(keep=False)].head(10))

clean = df.drop_duplicates().copy()

print("중복 제거 전 행 수:", len(df))
print("중복 제거 후 행 수:", len(clean))
print("제거된 행 수:", len(df) - len(clean))

### 왜 이 코드가 정답인지

중복 행은 같은 응답이 여러 번 들어간 상태다. 평균, 개수, 비율을 모두 왜곡할 수 있으므로 먼저 제거한다. `df.drop_duplicates().copy()` 로 원본은 보존하고 정제본을 만든다. `copy()` 를 붙이면 이후 `clean` 에 열을 추가하거나 값을 바꿀 때 원본과 섞이는 문제를 줄일 수 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 전체 행 기준 중복 | 8 |
| 중복 제거 후 행 수 | 520 |

---

## 문제 4 정답 — 날짜와 숫자형 타입 변환

In [ ]:
clean["submitted_at"] = pd.to_datetime(clean["submitted_at"], errors="coerce")

numeric_cols = ["satisfaction", "study_hours", "screen_time", "score"]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

print("변환 후 dtypes:")
print(clean.dtypes)
print("날짜 변환 실패 개수:", clean["submitted_at"].isna().sum())
print("숫자형 변환 후 결측 개수:")
print(clean[numeric_cols].isna().sum())

### 왜 이 코드가 정답인지

날짜 문자열은 날짜형으로 변환해야 기간, 월, 요일 분석이 가능하다. 숫자처럼 보이는 열도 실제로는 문자열이나 비정상 값이 섞일 수 있으므로 `pd.to_numeric` 으로 확정한다. `errors="coerce"` 는 변환 불가능한 값을 결측치로 바꿔 문제를 숨기지 않게 한다. 변환 후에는 dtypes와 결측 개수를 다시 확인해야 한다.

**지도 메모**

학생이 `astype(float)` 만 사용하면 변환 불가능한 값에서 바로 오류가 날 수 있다. 정제 수업에서는 오류를 무시하기보다 결측으로 바꾸고 이후 처리 기준을 세우는 흐름을 우선한다.

---

## 문제 5 정답 — 범주형 값 정리

In [ ]:
print("정리 전 gender 분포:")
print(clean["gender"].value_counts(dropna=False))
print("정리 전 club 분포:")
print(clean["club"].value_counts(dropna=False))

clean["gender"] = clean["gender"].replace({"unknown": "U", "": "U"}).fillna("U")
clean["club"] = clean["club"].replace({"": "none"}).fillna("none")

print("정리 후 gender 분포:")
print(clean["gender"].value_counts(dropna=False))
print("정리 후 club 분포:")
print(clean["club"].value_counts(dropna=False))
print("gender/club 결측:")
print(clean[["gender", "club"]].isna().sum())

### 왜 이 코드가 정답인지

범주형 값은 무리하게 추정하면 안 된다. 성별이 `unknown` 이거나 결측이면 알 수 없다는 의미의 `"U"` 로 통일한다. 동아리는 결측과 빈 문자열을 `"none"` 으로 통일해 분석 가능한 범주로 만든다. 정리 전후 분포를 모두 출력해야 처리 결과가 의도와 맞는지 확인할 수 있다.

**채점 기준**

- `unknown` 을 임의로 `M` 또는 `F` 로 바꾸면 안 된다.
- 결측을 처리한 뒤 `value_counts(dropna=False)` 로 남은 결측을 확인해야 한다.
- 범주형 처리 기준을 주석이나 결론에 남겨야 한다.

---

## 문제 6 정답 — 숫자형 결측치 채우기

In [ ]:
missing_before_fill = clean[["satisfaction", "screen_time", "study_hours", "score"]].isna().sum()

satisfaction_median = clean["satisfaction"].median()
screen_time_median = clean["screen_time"].median()

# 숫자형 결측은 이상치 영향을 줄이기 위해 중앙값으로 채운다.
clean["satisfaction"] = clean["satisfaction"].fillna(satisfaction_median)
clean["screen_time"] = clean["screen_time"].fillna(screen_time_median)

missing_after_fill = clean[["satisfaction", "screen_time", "study_hours", "score"]].isna().sum()

print("채우기 전 결측:")
print(missing_before_fill)
print("satisfaction 중앙값:", satisfaction_median)
print("screen_time 중앙값:", screen_time_median)
print("채우기 후 결측:")
print(missing_after_fill)

### 왜 이 코드가 정답인지

`satisfaction` 과 `screen_time` 에 결측이 남아 있으므로 분석 전에 채워야 한다. 중앙값은 평균보다 이상치 영향을 덜 받는다. 이번 데이터에는 `study_hours` 와 `score` 결측은 없지만, 함께 확인하면 처리 누락을 막을 수 있다. 채우기 전후 결측 개수를 비교해야 처리가 실제로 적용되었는지 검증할 수 있다.

**예상 핵심값**

```text
satisfaction 중앙값: 3.5
screen_time 중앙값: 3.4
```

---

## 문제 7 정답 — 이상치 범위 진단

In [ ]:
score_outlier = (clean["score"] < 0) | (clean["score"] > 100)
study_outlier = (clean["study_hours"] < 0) | (clean["study_hours"] > 8)
screen_outlier = (clean["screen_time"] < 0) | (clean["screen_time"] > 12)

print("score 이상치 수:", score_outlier.sum())
print("study_hours 이상치 수:", study_outlier.sum())
print("screen_time 이상치 수:", screen_outlier.sum())

outlier_rows = clean[score_outlier | study_outlier | screen_outlier]
print("이상치 행 예시:")
print(outlier_rows.head(10))

print("정제 전 숫자형 요약:")
print(clean[["satisfaction", "study_hours", "screen_time", "score"]].describe())

### 왜 이 코드가 정답인지

점수는 0~100 범위, 하루 학습 시간은 0~8시간, 화면 시간은 0~12시간으로 기준을 둔다. 이 기준을 벗어난 값은 분석 전에 확인해야 한다. 이상치를 바로 삭제하지 않고 먼저 개수와 예시를 보는 이유는, 삭제할지 보정할지 기준을 세우기 위해서다. `describe()` 는 최솟값과 최댓값을 통해 범위 문제를 빠르게 보여준다.

**예상 핵심값**

| 열 | 이상치 수 |
|---|---:|
| score | 10 |
| study_hours | 14 |
| screen_time | 0 |

---

## 문제 8 정답 — 이상치 보정

In [ ]:
clean["score"] = clean["score"].clip(0, 100)
clean["study_hours"] = clean["study_hours"].clip(0, 8)
clean["screen_time"] = clean["screen_time"].clip(0, 12)
clean["satisfaction"] = clean["satisfaction"].clip(1, 5)

range_check = clean[["satisfaction", "study_hours", "screen_time", "score"]].agg(["min", "max"])
print("보정 후 범위:")
print(range_check)

### 왜 이 코드가 정답인지

`clip` 은 값이 너무 작으면 하한으로, 너무 크면 상한으로 제한한다. 점수 130은 100으로, 점수 -5는 0으로 바뀐다. 학습 시간 15시간, 18시간 같은 값은 하루 수업 설문 기준에서 비현실적이므로 8시간으로 제한한다. 삭제 대신 보정을 택하면 응답 자체는 유지하면서 극단값이 평균을 왜곡하는 것을 줄일 수 있다.

**주의할 점**

이상치 보정 기준은 상황에 따라 달라질 수 있다. 중요한 것은 기준을 숨기지 않고 코드와 결론에 남기는 것이다.

---

## 문제 9 정답 — 품질 이슈 플래그 만들기

In [ ]:
raw_after_dedup = df.drop_duplicates().copy()
raw_after_dedup["satisfaction"] = pd.to_numeric(raw_after_dedup["satisfaction"], errors="coerce")
raw_after_dedup["study_hours"] = pd.to_numeric(raw_after_dedup["study_hours"], errors="coerce")
raw_after_dedup["screen_time"] = pd.to_numeric(raw_after_dedup["screen_time"], errors="coerce")
raw_after_dedup["score"] = pd.to_numeric(raw_after_dedup["score"], errors="coerce")

clean["had_missing"] = raw_after_dedup.isna().any(axis=1).to_numpy()
clean["score_outlier"] = ((raw_after_dedup["score"] < 0) | (raw_after_dedup["score"] > 100)).to_numpy()
clean["study_outlier"] = ((raw_after_dedup["study_hours"] < 0) | (raw_after_dedup["study_hours"] > 8)).to_numpy()

clean["quality_issue_count"] = (
    clean["had_missing"].astype(int)
    + clean["score_outlier"].astype(int)
    + clean["study_outlier"].astype(int)
)

print("품질 이슈 개수 분포:")
print(clean["quality_issue_count"].value_counts().sort_index())
print(clean[["student_id", "had_missing", "score_outlier", "study_outlier", "quality_issue_count"]].head())

### 왜 이 코드가 정답인지

정제 후에는 결측과 이상치가 사라지므로, 원래 문제가 있었는지 알기 어렵다. 그래서 중복 제거 직후의 원본 상태를 기준으로 플래그를 만든다. `had_missing` 은 원래 결측이 있었는지, `score_outlier` 와 `study_outlier` 는 범위 문제가 있었는지 기록한다. `quality_issue_count` 는 한 응답이 가진 품질 이슈 개수를 합산한 값이다.

**지도 메모**

학생이 정제 후 값으로 플래그를 만들면 대부분 False가 되어 버린다. "정제 전 진단값을 플래그로 남긴다"는 흐름을 강조한다.

---

## 문제 10 정답 — 응답 품질 점수 만들기

In [ ]:
clean["quality_score"] = 100 - clean["quality_issue_count"] * 20
clean["quality_score"] = clean["quality_score"].clip(0, 100)

print("품질 점수 평균:", f"{clean['quality_score'].mean():.2f}")
print("품질 점수 중앙값:", f"{clean['quality_score'].median():.2f}")
print("품질 점수 최솟값:", clean["quality_score"].min())
print("품질 100점 응답 수:", (clean["quality_score"] == 100).sum())
print("품질 80점 미만 예시:")
print(clean[clean["quality_score"] < 80].head())

### 왜 이 코드가 정답인지

품질 점수는 행별 데이터 신뢰도를 단순화한 보조 지표다. 이슈가 하나 있으면 20점 감점하는 기준을 적용하면 결측, 점수 이상치, 학습 시간 이상치를 같은 틀로 볼 수 있다. `clip(0, 100)` 은 감점 결과가 범위를 벗어나지 않게 한다. 품질 점수는 학생의 성취도가 아니라 데이터 정제 관점의 신뢰도라는 점을 분명히 해야 한다.

**채점 포인트**

- 품질 점수의 기준을 코드 또는 문장으로 설명했는가.
- 평균, 중앙값, 최솟값, 100점 응답 수를 출력했는가.
- 품질 점수와 성적 점수를 혼동하지 않았는가.

---

## 문제 11 정답 — 정제 완료 검증

In [ ]:
print("정제 후 결측:")
print(clean.isna().sum())

print("정제 후 숫자형 요약:")
print(clean[["satisfaction", "study_hours", "screen_time", "score", "quality_score"]].describe())

score_ok = clean["score"].between(0, 100).all()
study_ok = clean["study_hours"].between(0, 8).all()
screen_ok = clean["screen_time"].between(0, 12).all()
satisfaction_ok = clean["satisfaction"].between(1, 5).all()
missing_ok = clean.isna().sum().sum() == 0

is_clean = score_ok and study_ok and screen_ok and satisfaction_ok and missing_ok

print("최종 shape:", clean.shape)
print("score 범위 정상:", score_ok)
print("study_hours 범위 정상:", study_ok)
print("screen_time 범위 정상:", screen_ok)
print("satisfaction 범위 정상:", satisfaction_ok)
print("정제 완료:", is_clean)

### 왜 이 코드가 정답인지

정제는 처리보다 검증이 중요하다. 결측을 채웠다면 결측 개수가 0인지 확인해야 하고, 이상치를 보정했다면 범위 조건을 다시 확인해야 한다. `between(...).all()` 은 모든 값이 기준 범위 안에 있는지 Boolean 값으로 알려준다. 여러 조건을 묶은 `is_clean` 은 최종 검증 결과를 한눈에 보여준다.

**예상 핵심값**

정제 후 결측은 0개이고, `score`, `study_hours`, `screen_time`, `satisfaction` 은 모두 기준 범위 안에 있어야 한다.

---

## 문제 12 정답 — 학년별 정제 결과 요약

In [ ]:
grade_summary = clean.groupby("grade").agg(
    responses=("student_id", "count"),
    avg_score=("score", "mean"),
    avg_study=("study_hours", "mean"),
    avg_screen=("screen_time", "mean"),
    avg_satisfaction=("satisfaction", "mean"),
    avg_quality=("quality_score", "mean"),
).sort_values("avg_score", ascending=False)

print("학년별 요약:")
print(grade_summary)
print("평균 점수 1위 학년:", grade_summary["avg_score"].idxmax())

### 왜 이 코드가 정답인지

정제된 데이터로 학년별 응답 수와 평균 지표를 계산하면 학년 간 차이를 볼 수 있다. 정제 전 데이터에는 중복, 결측, 이상치가 있으므로 요약 결과가 왜곡될 수 있다. `clean.groupby("grade").agg(...)` 는 정제 후 데이터를 기준으로 학년별 평균 점수, 학습 시간, 화면 시간, 만족도, 품질 점수를 한 번에 계산한다.

**예상 해석**

평균 점수는 9학년이 가장 높게 나올 수 있다. 다만 학년별 차이가 크지 않으므로 결론에서는 과장하지 않는다.

---

## 문제 13 정답 — 동아리별 정제 결과 요약

In [ ]:
club_summary = clean.groupby("club").agg(
    responses=("student_id", "count"),
    avg_score=("score", "mean"),
    avg_study=("study_hours", "mean"),
    avg_quality=("quality_score", "mean"),
).sort_values("avg_score", ascending=False)

print("동아리별 요약:")
print(club_summary)
print("평균 점수 1위 동아리:", club_summary["avg_score"].idxmax())
print("응답 수 1위 동아리:", club_summary["responses"].idxmax())

### 왜 이 코드가 정답인지

동아리별 요약은 범주형 열을 기준으로 정제 결과를 비교하는 문제다. `club` 에서 결측을 `"none"` 으로 통일했기 때문에 미응답 또는 동아리 없음도 하나의 그룹으로 분석할 수 있다. 평균 점수 1위와 응답 수 1위가 다를 수 있으므로 두 기준을 분리해 출력한다.

**예상 핵심값**

```text
평균 점수 1위 동아리: math
응답 수 1위 동아리: none
```

---

## 문제 14 정답 — 정제 데이터 저장

In [ ]:
clean.to_csv(OUTPUT_PATH, index=False)

print("저장 경로:", OUTPUT_PATH)
print("파일 생성 확인:", os.path.exists(OUTPUT_PATH))

reloaded = pd.read_csv(OUTPUT_PATH)
print("저장 전 shape:", clean.shape)
print("다시 읽은 shape:", reloaded.shape)
print(reloaded.head(3))

### 왜 이 코드가 정답인지

정제 결과는 다른 분석이나 보고서에서 다시 사용할 수 있도록 저장한다. `index=False` 를 사용하면 pandas 인덱스가 불필요한 새 컬럼으로 저장되지 않는다. 저장 후 다시 읽어 shape 를 비교하면 파일이 제대로 만들어졌는지 확인할 수 있다. 원본 `dirty_survey.csv` 를 덮어쓰지 않고 별도 파일로 저장하는 것이 안전하다.

**채점 포인트**

- 원본 파일을 덮어쓰지 않았는가.
- 저장 파일 생성 여부를 확인했는가.
- 다시 읽은 파일의 행 수와 열 수를 비교했는가.

---

## 문제 15 정답 — 설문 데이터 클린업 + 품질 점수 결론

In [ ]:
original_rows = len(df)
clean_rows = len(clean)
removed_duplicates = original_rows - clean_rows
original_missing = df.isna().sum().sum()
clean_missing = clean.isna().sum().sum()
avg_quality = clean["quality_score"].mean()
perfect_quality_count = (clean["quality_score"] == 100).sum()

print("원본 행 수:", original_rows)
print("정제 후 행 수:", clean_rows)
print("제거한 중복 행 수:", removed_duplicates)
print("원본 결측 셀 수:", original_missing)
print("정제 후 결측 셀 수:", clean_missing)
print("평균 품질 점수:", f"{avg_quality:.2f}")
print("품질 100점 응답 수:", perfect_quality_count)

### 왜 이 코드가 정답인지

최종 결론에는 정제 전후의 핵심 변화가 들어가야 한다. 원본 행 수와 정제 후 행 수는 중복 제거 결과를 보여주고, 결측 셀 수 변화는 결측 처리 결과를 보여준다. 평균 품질 점수와 품질 100점 응답 수는 정제 전 문제가 얼마나 있었는지 요약한다. 이 숫자를 결론에 연결해야 코드 결과가 보고서가 된다.

**결론 예시**

```text
원본 설문 데이터는 528행이었고, 중복 8행을 제거해 최종 520행으로 정리했다.
원본에는 결측 셀이 244개 있었지만, 범주형은 U/none으로 통일하고 숫자형은 중앙값으로 채워 정제 후 결측을 0개로 만들었다.
점수는 0~100, 학습 시간은 0~8시간, 화면 시간은 0~12시간으로 보정해 극단값 영향을 줄였다.
다음 분석에서는 품질 점수가 낮은 응답을 따로 검토하고, 학년별·동아리별 평균 차이가 실제 의미 있는지 추가 확인하겠다.
```

---

## 전체 채점 메모

| 구간 | 문제 | 핵심 개념 | 필수 통과 조건 |
|---|---|---|---|
| 진단 | 1~2 | 구조, 결측 | 원본 크기와 결측 현황 확인 |
| 원본 보존 | 3~4 | 중복 제거, 타입 변환 | `df` 보존, `clean` 생성 |
| 결측 처리 | 5~6 | 범주형/숫자형 처리 | U/none, 중앙값 기준 설명 |
| 이상치 처리 | 7~8 | 범위 진단, clip | 처리 전 개수와 처리 후 범위 확인 |
| 품질 지표 | 9~10 | 플래그, 품질 점수 | 정제 전 이슈 기준으로 점수 생성 |
| 검증/저장 | 11~15 | 최종 검증, 저장, 결론 | 결측 0개, 범위 정상, CSV 저장 |

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 교사 피드백 |
|---|---|---|
| 원본 `df` 를 바로 수정 | 정제 전후 비교 불가 | `clean = df.drop_duplicates().copy()` 로 분리 |
| 결측을 모두 0으로 채움 | 타입별 의미 무시 | 숫자형/범주형 기준을 분리 |
| `unknown` 을 M/F로 바꿈 | 임의 추정 | 모르는 값은 U로 유지 |
| 이상치를 먼저 clip하고 개수 확인 | 진단 기록 손실 | 처리 전 이상치 개수를 먼저 저장 |
| 품질 점수를 성적처럼 해석 | 지표 의미 혼동 | 데이터 신뢰도 점수라고 다시 안내 |
| 저장 파일이 원본과 같은 이름 | 원본 훼손 위험 | 별도 출력 파일 사용 |

## 부분 점수 운영 기준

1. 문제 1~2에서 진단 없이 바로 처리한 답안은 코드가 맞아도 보충 설명을 요구한다.
2. 문제 3에서 원본을 보존하지 않았더라도 결과가 맞으면 부분 통과 가능하지만, 정제 전후 비교가 어려운 점을 피드백한다.
3. 문제 5~6 결측 처리 기준이 다를 수는 있다. 단, 기준을 설명하고 처리 후 결측을 확인해야 한다.
4. 문제 7~8에서 이상치를 삭제한 경우도 논리적으로 설명하면 부분 인정 가능하지만, 이번 모범 기준은 `clip` 이다.
5. 문제 14 저장을 못 했더라도 정제 결과가 완성되어 있으면 코드 점수는 인정하고, 파일 저장은 보충하게 한다.
6. 문제 15 결론에 숫자가 없으면 분석 제출물로는 미완성이다.

## 문제별 지도 질문

| 문제 | 학생에게 던질 질문 | 확인할 답 |
|---:|---|---|
| 1 | 원본 행 수와 열 수는 왜 먼저 기록하나요? | 정제 후 비교 기준 |
| 2 | 결측 개수와 결측 비율은 어떻게 다른가요? | 절대량과 전체 대비 비중 |
| 3 | 중복을 제거하면 어떤 통계가 달라질 수 있나요? | 평균, 개수, 비율 |
| 4 | `errors="coerce"` 는 어떤 역할을 하나요? | 변환 실패를 결측으로 표시 |
| 5 | `unknown` 을 U로 둔 이유는 무엇인가요? | 임의 추정을 피함 |
| 6 | 중앙값으로 채우는 이유는 무엇인가요? | 이상치 영향을 줄임 |
| 7 | 이상치를 삭제하기 전에 무엇을 확인해야 하나요? | 개수와 예시 |
| 8 | `clip` 은 값을 어떻게 바꾸나요? | 하한/상한 밖 값을 경계값으로 제한 |
| 9 | 플래그는 정제 전 기준으로 만들어야 하는 이유는 무엇인가요? | 정제 후에는 이슈 흔적이 사라짐 |
| 10 | 품질 점수는 무엇을 의미하나요? | 데이터 신뢰도 |
| 11 | 정제 완료는 어떤 조건으로 판단하나요? | 결측 0개, 범위 정상 |
| 12 | 정제 후 학년별 요약을 보는 이유는 무엇인가요? | 왜곡이 줄어든 비교 |
| 13 | `none` 그룹은 어떻게 해석해야 하나요? | 미응답/동아리 없음 |
| 14 | `index=False` 는 왜 쓰나요? | 불필요한 인덱스 컬럼 저장 방지 |
| 15 | 좋은 결론에는 무엇이 들어가야 하나요? | 숫자, 처리 기준, 다음 행동 |

## 보충 설명 포인트

- 정제는 "예쁜 데이터 만들기"가 아니라 분석 기준을 명확히 하는 과정이다. 결측을 어떻게 채웠는지, 이상치를 왜 보정했는지 설명하지 못하면 정제 결과를 신뢰하기 어렵다.
- 범주형 결측과 숫자형 결측은 다르게 처리한다. 성별을 평균으로 채울 수 없고, 학습 시간을 `"unknown"` 문자열로 채우는 것도 분석에 맞지 않다.
- 이상치는 항상 나쁜 값이 아니다. 하지만 이번 데이터처럼 점수 130점, 하루 학습 18시간처럼 기준을 벗어난 값은 분석 목적에 맞게 보정한다.
- 품질 점수는 간단한 교육용 지표다. 실제 프로젝트에서는 이슈 종류별 가중치를 다르게 두거나, 품질 낮은 행을 별도 검토 대상으로 분리할 수 있다.

## 재실행 확인 순서

1. 런타임을 새로 시작한다.
2. 환경 셀부터 문제 15까지 순서대로 실행한다.
3. `clean.isna().sum().sum()` 이 0인지 확인한다.
4. `score`, `study_hours`, `screen_time`, `satisfaction` 범위가 기준 안에 있는지 확인한다.
5. `clean_survey.csv` 가 생성되었는지 확인하고, 원본 파일을 덮어쓰지 않았는지 확인한다.

학생 답안은 정답 코드와 달라도 된다. 다만 정제 기준, 처리 전 진단, 처리 후 검증, 결론의 숫자가 모두 연결되어야 통과로 본다.